# Bondview Duration — Persistence vs. Hysteresis: Transition Lag and State Carryover

**Research diagnostic, not a change to authoritative design.** This is a self-contained follow-up to
`bondview_duration_trend_6m_stabilization_review.ipynb`. It preserves that notebook unchanged.

**Authoritative starting point:** `lbk209/bondview-v2`, branch `update_duration_plan`,
commit `42c6465018e98eb8a8dc8e7ab1c1c7114d55af67` (observed 2026-10-09).
The input is `validation/261002_duration/task1_core/results.csv` (Git blob
`793ded82a7c3367d79ae4fbe7a700dd3d1b583bf`). Results were independently
calculated using the connector-accessible full source CSV. Re-running these cells
in the checked-out repository reproduces the diagnostics from the original data.

The current working package is **6M overlapping Trend = M5(t) − M5(t−126)**;
5-observation endpoint smoothing, **±25 bp directional entry** for all three
representative tenors, and **±10 bp Recent Move**. The earlier 3M Trend preference
has been superseded. The **5 bp hysteresis rule remains provisional** pending
human review; this notebook neither changes it nor optimizes any parameters.

## Questions addressed

1. Does persistence's 1–2-observation transition delay actually matter for durable Trend transitions?
2. How often does hysteresis retain a directional State while a plain ±25 bp classifier
   would say Stable, and for how long?
3. Does hysteresis sometimes bridge a Stable interval so that a renewed directional run
   is not a *new* transition under hysteresis?
4. As a secondary diagnostic, how long do opposing **Trend × Recent Move** Rule Cases
   persist under plain classification versus 5 bp hysteresis?

**Interpretation limitation:** “durable” here means a *retrospectively observed*
plain-State run of at least 10 or 21 valid observations. It is not a market-truth label,
a forecast, a transaction-cost test, or an ETF-return validation. All statistics are
computed inside contiguous accepted source segments; no structural gap is bridged.

## 1. Environment, source lineage, and fixed definitions

Place this notebook anywhere **inside** a checked-out Bondview v2 repository, or
run Jupyter from that repository. The repository's CSV is a prerequisite; this
notebook intentionally does not embed or download market history. No source files
are modified and no new Raw Observations are fetched.

In [ ]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

EXPECTED_COMMIT = "42c6465018e98eb8a8dc8e7ab1c1c7114d55af67"
CSV_REL = Path("validation/261002_duration/task1_core/results.csv")
SOURCE_NOTEBOOK = "bondview_duration_trend_6m_stabilization_review.ipynb"
EXPECTED_CLASSIFIED_N = 38330


def locate_repo_root():
    here = Path.cwd().resolve()
    for path in (here, *here.parents, Path("/home/lbk/works/bondview-v2")):
        if (path / CSV_REL).is_file():
            return path
    raise FileNotFoundError(
        f"Unable to find {CSV_REL}. Open/run this notebook from a local "
        "checkout of the update_duration_plan branch."
    )

REPO_ROOT = locate_repo_root()
TASK1_PATH = REPO_ROOT / CSV_REL

def git_command(*args):
    try:
        return subprocess.check_output(["git", "-C", str(REPO_ROOT), *args], text=True, stderr=subprocess.DEVNULL).strip()
    except (OSError, subprocess.CalledProcessError):
        return "not available"

print("Repository root:", REPO_ROOT)
print("Source CSV:", TASK1_PATH)
print("Git branch:", git_command("branch", "--show-current"))
print("Current HEAD:", git_command("rev-parse", "HEAD"))
print("Expected branch snapshot:", EXPECTED_COMMIT)

In [ ]:
source = pd.read_csv(TASK1_PATH, parse_dates=["as_of"])
required = {
    "as_of", "representative_tenor", "segment", "observation_index",
    "trend_bp", "trend_state", "recent_move_state",
}
assert required.issubset(source.columns), sorted(required - set(source.columns))
review = (
    source.loc[source["trend_bp"].notna() & source["trend_state"].notna() &
               source["segment"].notna(), list(required)]
    .copy()
    .sort_values(["representative_tenor", "segment", "as_of"])
    .reset_index(drop=True)
)
review["segment"] = review["segment"].astype(int)
review["trend_bp"] = review["trend_bp"].astype(float)
assert len(review) == EXPECTED_CLASSIFIED_N, (len(review), EXPECTED_CLASSIFIED_N)
assert review["recent_move_state"].notna().all()
for (tenor, segment), g in review.groupby(["representative_tenor", "segment"], sort=False):
    assert g["observation_index"].diff().iloc[1:].eq(1).all(), (tenor, segment)

print("Classified observations:", len(review))
display(review.groupby("representative_tenor").size().rename("n").to_frame())

## 2. Classifiers and synthetic regression tests

The functions below reproduce the **exact transition semantics** of the existing
6M stabilization notebook, including direct transitions through the opposite
±25 bp entry threshold and initial State handling at each structural segment.

The two paths `26 → 24 → 24 → 19` and `24 → 24 → 24 → 19` illustrate hysteresis's
path dependence. To provide an explicit initial Stable State, the regression test
prepends `24` to each sequence.

In [ ]:
def classify_plain(value, entry=25.0):
    if value < -entry:
        return "Falling"
    if value > entry:
        return "Rising"
    return "Stable"


def apply_persistence(raw_states, required_n):
    raw_states = list(raw_states)
    if not raw_states:
        return []
    active = raw_states[0]
    pending, count = None, 0
    out = [active]
    for candidate in raw_states[1:]:
        if candidate == active:
            pending, count = None, 0
        else:
            if candidate == pending:
                count += 1
            else:
                pending, count = candidate, 1
            if count >= required_n:
                active, pending, count = pending, None, 0
        out.append(active)
    return out


def apply_hysteresis(values, entry_bp=25.0, exit_bp=20.0):
    values = list(values)
    if not values:
        return []
    active = classify_plain(values[0], entry_bp)
    out = [active]
    for value in values[1:]:
        if active == "Stable":
            active = classify_plain(value, entry_bp)
        elif active == "Rising":
            if value < -entry_bp:
                active = "Falling"
            elif value <= exit_bp:
                active = "Stable"
        elif active == "Falling":
            if value > entry_bp:
                active = "Rising"
            elif value >= -exit_bp:
                active = "Stable"
        out.append(active)
    return out

VARIANTS = {
    "plain": lambda values, raw: list(raw),
    "persistence_2": lambda values, raw: apply_persistence(raw, 2),
    "persistence_3": lambda values, raw: apply_persistence(raw, 3),
    "hysteresis_5bp": lambda values, raw: apply_hysteresis(values, 25.0, 20.0),
    "hysteresis_10bp": lambda values, raw: apply_hysteresis(values, 25.0, 15.0),
}

path_a = [24, 26, 24, 24, 19]
path_b = [24, 24, 24, 24, 19]
assert apply_hysteresis(path_a) == ["Stable", "Rising", "Rising", "Rising", "Stable"]
assert apply_hysteresis(path_b) == ["Stable"] * 5
assert apply_persistence([classify_plain(x) for x in path_a], 2) == ["Stable"] * 5
path_c = [24, 26, 27, 27, 27]
assert apply_persistence([classify_plain(x) for x in path_c], 2) == ["Stable", "Stable", "Rising", "Rising", "Rising"]
assert apply_persistence([classify_plain(x) for x in path_c], 3) == ["Stable", "Stable", "Stable", "Rising", "Rising"]
assert classify_plain(25) == classify_plain(-25) == "Stable"
print("Synthetic regression tests: passed")

In [ ]:
review["plain_state"] = review["trend_bp"].map(classify_plain)
assert (review["plain_state"] == review["trend_state"]).all(), "Task-1 baseline mismatch"

frames = []
for (_, _), group in review.groupby(["representative_tenor", "segment"], sort=False):
    group = group.copy()
    values = group["trend_bp"].tolist()
    raw = group["plain_state"].tolist()
    for name, transform in VARIANTS.items():
        group[f"state_{name}"] = transform(values, raw)
    frames.append(group)
review = pd.concat(frames, ignore_index=True)
assert len(review) == EXPECTED_CLASSIFIED_N
print("Plain-State reproduction and segment-specific stabilization: passed")

## 3. Durable-transition recognition lag (10- and 21-observation horizons)

For each **plain-State** episode lasting at least the specified number of valid
observations, start at its first observation and count observations until the
other classifier first outputs that same State *within that episode*.

`lag_obs = 0` means no delay; a missing match is reported as `missed`, **not**
artificially assigned the episode length. Transitions are split into:

- **Entry:** Stable → Rising/Falling
- **Exit:** Rising/Falling → Stable
- **Reversal:** Rising → Falling or Falling → Rising (if any)

`previous_aligned=False` identifies when the alternate classifier's State just
before the plain transition was *already different* from the previous plain
State. This matters particularly for hysteresis: it may have stayed directional
through the preceding plain-Stable run, so a plain “new entry” need not be a new
hysteresis transition. Show such cases rather than counting them as clean entries.

In [ ]:
def plain_episodes(data):
    records = []
    for (tenor, segment), g in data.groupby(["representative_tenor", "segment"], sort=False):
        g = g.sort_values("as_of").reset_index(drop=True)
        states = g["state_plain"].tolist()
        start = 0
        while start < len(g):
            stop = start + 1
            while stop < len(g) and states[stop] == states[start]:
                stop += 1
            records.append((tenor, segment, g, start, stop, states[start],
                            states[start - 1] if start else None))
            start = stop
    return records

lag_records = []
for tenor, segment, g, start, stop, target, previous in plain_episodes(review):
    if previous is None:
        continue   # no observed transition at the left-censored segment boundary
    transition_type = ("exit" if target == "Stable" else
                       "entry" if previous == "Stable" else "reversal")
    for min_run in (10, 21):
        if stop - start < min_run:
            continue
        for method in ("persistence_2", "persistence_3", "hysteresis_5bp", "hysteresis_10bp"):
            path = g[f"state_{method}"].to_numpy()
            hits = np.flatnonzero(path[start:stop] == target)
            lag_records.append({
                "tenor": tenor, "segment": segment, "min_run": min_run,
                "transition": transition_type, "variant": method,
                "plain_start": g["as_of"].iat[start],
                "plain_end": g["as_of"].iat[stop - 1],
                "from_state": previous, "target_state": target,
                "plain_run_obs": stop - start,
                "lag_obs": int(hits[0]) if len(hits) else np.nan,
                "missed": not bool(len(hits)),
                "previous_aligned": path[start - 1] == previous,
                "already_target_before_start": path[start - 1] == target,
            })

lag_events = pd.DataFrame(lag_records)
assert not lag_events.empty

def aggregate_lags(events):
    successes = events["lag_obs"].dropna()
    return pd.Series({
        "n": len(events),
        "missed": int(events["missed"].sum()),
        "prior_state_not_aligned": int((~events["previous_aligned"]).sum()),
        "mean_lag": successes.mean(),
        "median_lag": successes.median(),
        "p90_lag": successes.quantile(.90),
        "max_lag": successes.max(),
    })

lag_summary = (
    lag_events.groupby(["tenor", "min_run", "transition", "variant"], sort=False)
    .apply(aggregate_lags, include_groups=False)
    .reset_index()
)
show_lags = lag_summary.loc[
    lag_summary["min_run"].eq(21)
    & lag_summary["variant"].isin(["persistence_2", "persistence_3", "hysteresis_5bp"])
    & lag_summary["transition"].isin(["entry", "exit"])
].copy()
display(show_lags.round(2))

In [ ]:
# Compare genuinely new hysteresis entries with plain episodes where the prior
# hysteresis State was already directional (a countertrend pause was bridged).
entry_rearms = (
    lag_events.loc[(lag_events["min_run"] == 10) &
                   (lag_events["transition"] == "entry") &
                   (lag_events["variant"] == "hysteresis_5bp")]
    .groupby("tenor")
    .agg(plain_durable_entries=("lag_obs", "size"),
         genuinely_rearmed=("previous_aligned", "sum"),
         inherited_same_direction=("already_target_before_start", "sum"))
)
assert (entry_rearms["genuinely_rearmed"] +
        entry_rearms["inherited_same_direction"] ==
        entry_rearms["plain_durable_entries"]).all()
display(entry_rearms)

missed_ten = lag_events.loc[
    (lag_events["variant"] == "hysteresis_5bp") & lag_events["missed"]
    & lag_events["min_run"].eq(10)
]
display(missed_ten[["tenor", "transition", "plain_start", "plain_end", "plain_run_obs",
                    "from_state", "target_state"]])

## 4. State carryover *inside* the original ±25 bp Stable band

Carryover occurs when a tested variant says `Falling`/`Rising` while plain
classification says `Stable`. Count maximal consecutive episodes, separately by
source segment. Report median, upper percentile, maximum, total observation count,
and the fraction of all classified observations. These are not return forecasts.

The `hysteresis_5bp` mechanism is constrained to carry a Rising/Falling State only
within the **20–25 bp strip** next to its own entry boundary; it cannot carry the
State into an arbitrarily deep Stable band. Yet it can remain in that strip for
many successive observations. Persistence carryover instead arises because a new
Stable classification awaits 2 or 3 observations of confirmation.

In [ ]:
carry_records = []
for (tenor, segment), g in review.groupby(["representative_tenor", "segment"], sort=False):
    g = g.sort_values("as_of").reset_index(drop=True)
    for variant in ("persistence_2", "persistence_3", "hysteresis_5bp", "hysteresis_10bp"):
        s = g[f"state_{variant}"]
        carried = (s.ne("Stable") & g["state_plain"].eq("Stable"))
        run_ids = carried.ne(carried.shift()).cumsum()
        for _, block in g.loc[carried].groupby(run_ids.loc[carried]):
            carry_records.append({
                "tenor": tenor, "segment": segment, "variant": variant,
                "from_date": block["as_of"].iat[0],
                "to_date": block["as_of"].iat[-1],
                "direction": block[f"state_{variant}"].iat[0],
                "duration_obs": len(block),
                "min_trend_bp": block["trend_bp"].min(),
                "max_trend_bp": block["trend_bp"].max(),
            })
carryover = pd.DataFrame(carry_records)
assert not carryover.empty

obs_by_tenor = review.groupby("representative_tenor").size()
def aggregate_carryover(g):
    total_obs = obs_by_tenor.loc[g.name[0]]
    return pd.Series({
        "episodes": len(g), "total_obs": int(g["duration_obs"].sum()),
        "observation_pct": g["duration_obs"].sum() * 100 / total_obs,
        "mean_obs": g["duration_obs"].mean(),
        "median_obs": g["duration_obs"].median(),
        "p90_obs": g["duration_obs"].quantile(.9),
        "max_obs": g["duration_obs"].max(),
    })

carryover_summary = (
    carryover.groupby(["tenor", "variant"], sort=False)
    .apply(aggregate_carryover, include_groups=False)
    .reset_index()
)
display(carryover_summary.round(2))

print("Longest 5 bp hysteresis carryover episodes by tenor:")
display(
    carryover.loc[carryover["variant"].eq("hysteresis_5bp")]
    .sort_values(["tenor", "duration_obs"], ascending=[True, False])
    .groupby("tenor", sort=False).head(4).reset_index(drop=True)
)

# Independent anchor checks from the frozen historical data. If these fail,
# investigate the implementation or source before drawing conclusions.
anchors = {"6M": (77, 219, 13), "10Y": (190, 403, 11), "30Y": (148, 335, 9)}
for tenor, (episodes, days, maximum) in anchors.items():
    row = carryover_summary.loc[(carryover_summary["tenor"] == tenor) &
                                (carryover_summary["variant"] == "hysteresis_5bp")].iloc[0]
    assert (int(row["episodes"]), int(row["total_obs"]), int(row["max_obs"])) == (episodes, days, maximum)
print("Historical carryover anchor checks: passed")

## 5. Secondary diagnostic — opposing Trend × Recent Move episodes

These Rule Cases are **not** necessarily errors. In particular, a 6M broader
Rising-yield condition may coexist with a 1M Falling-yield relief move. Count
maximal consecutive runs of `Falling × Rising` and `Rising × Falling` inside each
source segment for the plain classifier and 5 bp hysteresis. We do not attempt to
minimize their frequency or infer how ETF prices would subsequently perform.

In [ ]:
opp_records = []
OPPOSING = {"Falling × Rising", "Rising × Falling"}
for (tenor, segment), g in review.groupby(["representative_tenor", "segment"], sort=False):
    g = g.sort_values("as_of").reset_index(drop=True)
    for variant in ("plain", "hysteresis_5bp"):
        labels = g[f"state_{variant}"] + " × " + g["recent_move_state"]
        runs = labels.ne(labels.shift()).cumsum()
        for _, block in g.groupby(runs):
            label = labels.loc[block.index[0]]
            if label in OPPOSING:
                opp_records.append({
                    "tenor": tenor, "segment": segment, "variant": variant,
                    "case": label, "n_obs": len(block),
                    "start": block["as_of"].iat[0], "end": block["as_of"].iat[-1],
                })
opposing = pd.DataFrame(opp_records)
opp_summary = (
    opposing.groupby(["tenor", "variant", "case"], sort=False)["n_obs"]
    .agg(episodes="size", mean="mean", median="median", p90=lambda v: v.quantile(.90), maximum="max")
    .reset_index()
)
display(opp_summary.round(2))

## 6. Visual case inspection

Use the following function to inspect the original numeric Trend Value and the
resulting discrete State trajectories. A 5 bp hysteresis State is allowed to be
Rising at a value of +24 bp *only if it previously crossed above +25 bp*.

The selected windows illustrate long State carryover (1993, 1999, 1965) and an
important genuine reversal (1980). Use separate single-figure plots for value and
State to avoid conflating the underlying continuous Trend with its classification.

In [ ]:
STATE_NUM = {"Falling": -1, "Stable": 0, "Rising": 1}

def inspect_window(tenor, start, end):
    w = review.loc[
        review["representative_tenor"].eq(tenor) &
        review["as_of"].between(pd.Timestamp(start), pd.Timestamp(end))
    ].sort_values("as_of")
    if w.empty:
        print("No classified observations in this window.")
        return
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(w["as_of"], w["trend_bp"], marker=".", label="6M Trend Value (bp)")
    for boundary in (-25, -20, 20, 25):
        ax.axhline(boundary, linestyle="--", linewidth=1, label=f"{boundary:+g} bp")
    ax.set(title=f"{tenor} — Continuous Trend Value: {start} to {end}", ylabel="bp")
    ax.grid(alpha=.25)
    ax.legend(loc="best", ncol=3)
    fig.autofmt_xdate()
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 3.5))
    for variant in ("plain", "persistence_3", "hysteresis_5bp"):
        ax.step(w["as_of"], w[f"state_{variant}"].map(STATE_NUM),
                where="post", label=variant, alpha=.75)
    ax.set(title=f"{tenor} — State classification: {start} to {end}",
           yticks=[-1, 0, 1], yticklabels=["Falling", "Stable", "Rising"])
    ax.grid(alpha=.25)
    ax.legend(loc="best")
    fig.autofmt_xdate()
    plt.show()

# Two informative examples; uncomment others for deeper inspection.
inspect_window("6M", "1993-05-03", "1993-06-15")
inspect_window("10Y", "1965-11-01", "1965-12-15")
# inspect_window("6M", "1999-07-01", "1999-08-05")
# inspect_window("10Y", "1980-06-01", "1980-11-30")
# inspect_window("30Y", "2009-10-20", "2009-11-30")

## 7. Recorded findings from the full historical run

**Reproducibility anchor:** With the 38,330-row source snapshot, all plain-State
labels match, and the 5 bp hysteresis carryover results are:

| Tenor | Carryover episodes | Median (observations) | P90 | Maximum | Total carryover observations |
|:--|--:|--:|--:|--:|--:|
| 6M | 77 | 2 | 6 | 13 | 219 |
| 10Y | 190 | 1 | 4 | 11 | 403 |
| 30Y | 148 | 1 | 5 | 9 | 335 |

Under the **21-observation minimum** sustained-State definition, all qualifying
plain transitions eventually matched the persistence-2, persistence-3, and
hysteresis-5 classifiers. Persistence-2 typically recognized them after one
observation; persistence-3 after two. Hysteresis-5 matched new directional
entries immediately, but directional exits had variable delays: roughly 1–2
observations at the median, with maxima 7 / 9 / 5 for 6M / 10Y / 30Y.

With the less restrictive **10-observation minimum**, hysteresis-5 did not
report `Stable` anywhere within **four** otherwise sustained plain-Stable runs:
6M (1993-05-14 to 1993-06-02; 1999-07-12 to 1999-07-26) and
10Y (1965-11-16 to 1965-12-01; 2006-10-11 to 2006-10-25).
All four stayed close to the original threshold and inside the 5 bp strip.

A further semantic detail: among plain directional entries lasting ≥10 observations,
hysteresis was *already in that direction just before the new plain entry* in
12 of 51 cases (6M), 21 of 152 (10Y), and 23 of 105 (30Y). These are not
ordinary zero-lag *new* signals; they show hysteresis bridging shorter
plain-Stable interruptions. The `previous_aligned` flag makes this explicit.

### Assessment

The experiment **does not establish that persistence's 1–2-observation lag is
economically unacceptable** for a six-month Trend. Hysteresis-5 remains a
reasonable provisional preference because it recognizes threshold crossings
without a mandatory entry wait and is effective against short boundary churn.
Its cost, however, is a history-dependent State that may survive several days,
occasionally as long as 9–13 observations, after plain classification has turned
Stable. Which interpretation is preferable is a **Component-level semantic
choice**, not a statistical optimization of minimum episode length.

**Not done:** no ETF return comparison, no Core ordinal remapping, no sensitivity
search across new threshold values, no 3M rerun, and no design-document edits.

## 8. Optional inspection / export (disabled by default)

Running the notebook displays all outputs. To save the computed tables locally,
change `EXPORT = True` in the cell below. They are written to a *new directory*
beside this notebook's working directory, rather than overwriting any repository
source artifact.

In [ ]:
EXPORT = False
if EXPORT:
    output_dir = Path.cwd() / "bondview_stabilization_tradeoff_outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    for filename, table in (
        ("durable_transition_events.csv", lag_events),
        ("durable_transition_summary.csv", lag_summary),
        ("carryover_episodes.csv", carryover),
        ("carryover_summary.csv", carryover_summary),
        ("opposing_rule_case_episodes.csv", opposing),
        ("opposing_rule_case_summary.csv", opp_summary),
    ):
        table.to_csv(output_dir / filename, index=False)
    print("Exported to", output_dir)
else:
    print("No files written; set EXPORT = True to save diagnostic tables.")